# 01 — First inference

Load Gemma 4 E2B locally on MPS and generate with and without thinking mode.

**Multi-turn rule:** when feeding history back to the model, use `result.text` only — never re-include `result.thought`.

In [ ]:
from gemma4_lab.config import Settings
from gemma4_lab.observability import setup
from gemma4_lab.inference.hf_local import GemmaLocal

setup()  # logfire bootstrap — secrets read from os.environ inside the call
settings = Settings()
model = GemmaLocal(settings)
settings.model_id

## Plain generation (no thinking)

In [ ]:
result = model.generate(
    messages=[{"role": "user", "content": "Explain attention in one short paragraph."}],
    thinking=False,
    max_new_tokens=200,
)
print(result.text)
print(f"\n[in={result.input_tokens} out={result.output_tokens} {result.latency_ms:.0f}ms]")

## With thinking enabled

`result.thought` holds the internal reasoning channel; `result.text` is the visible answer.

In [ ]:
result = model.generate(
    messages=[{"role": "user", "content": "What is 17 * 23? Show your working."}],
    thinking=True,
    max_new_tokens=400,
)
print("THOUGHT:\n", result.thought)
print("\nANSWER:\n", result.text)

### Multi-turn — strip thoughts before re-prompting

Below shows the correct way to extend a conversation: only `result.text` (not `result.thought`) is fed back as the assistant turn.

In [ ]:
history = [
    {"role": "user", "content": "What is 17 * 23?"},
    {"role": "assistant", "content": result.text},  # NOT result.thought
    {"role": "user", "content": "Now divide that by 7."},
]
follow_up = model.generate(messages=history, thinking=False, max_new_tokens=80)
print(follow_up.text)